---
# Phase 3 — Anomaly Detection Modelling & Evaluation
## AIoT-Sol Dataset (6G) — Autoencoder & Isolation Forest

---

### Pipeline Overview

| Step | Description |
|---|---|
| **0** | Imports & configuration |
| **1** | Data loading, realistic anomaly simulation, stratified splits |
| **2** | Model selection justification |
| **3** | Autoencoder — architecture, training, threshold calibration on **held-out val set** |
| **4** | Isolation Forest — training with correct contamination strategy |
| **5** | Cross-validation stability analysis (both models) |
| **6** | Full comparative evaluation & visualisations |
| **7** | Model persistence (`.pkl` / `.keras`) |
| **8** | Critical analysis & recommendations |

### Key methodological fixes in this version
- **Realistic anomaly injection** — attack-type-specific perturbations instead of uniform Gaussian noise
- **No threshold leakage** — AE threshold calibrated on a dedicated validation split, never on the test set
- **Correct IF contamination** — always `'auto'` when training set is 100% benign; contamination estimated from held-out val set proportions
- **Cross-validation** — 5-fold StratifiedKFold stability analysis for both models
- **Full model persistence** — all artefacts saved and reload-verified


---
## Section 0 — Imports & Configuration

In [ ]:
# ============================================================
# SECTION 0 : IMPORTS & CONFIGURATION
# ============================================================
import os, pickle, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 120

MODELS_DIR = 'saved_models'
os.makedirs(MODELS_DIR, exist_ok=True)

print('✓ All imports OK')
print(f'  TensorFlow : {tf.__version__}')
print(f'  Pandas     : {pd.__version__}')
print(f'  NumPy      : {np.__version__}')
print(f'  Models will be saved to: {MODELS_DIR}/')


---
## Section 1 — Data Preparation

### 1.1 Load the Cleaned Dataset


In [ ]:
# ============================================================
# SECTION 1.1 : LOAD CLEANED DATASET
# ============================================================
df = pd.read_csv('AIoT_6G_CLEANED.csv')

print('=' * 60)
print('DATASET LOADED')
print('=' * 60)
print(f'  Shape    : {df.shape}')
print(f'  Columns  : {df.shape[1]}  ({df.shape[1]-1} features + Label)')
print()

label_counts = df['Label'].value_counts()
print('Label distribution:')
print(label_counts)
print()
benign_pct = label_counts.get('Benign', label_counts.iloc[0]) / len(df) * 100
print(f'  Benign  : {benign_pct:.1f}%')
print(f'  Attack  : {100-benign_pct:.1f}%')


### 1.2 Class Distribution Visualisation

In [ ]:
# ============================================================
# SECTION 1.2 : CLASS DISTRIBUTION VISUALISATION
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

label_counts.plot(kind='bar', ax=axes[0],
                  color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution (count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of flows')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

axes[1].pie(label_counts.values, labels=label_counts.index,
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=140, textprops={'fontsize': 12})
axes[1].set_title('Class Proportion (%)', fontsize=13, fontweight='bold')

plt.suptitle('AIoT-6G: Class Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()


---
### 1.3 Realistic Attack-Specific Anomaly Simulation

**Fix applied — replacing uniform Gaussian noise with structured attack-type perturbations.**

The original approach perturbed all 53 features simultaneously with 3σ Gaussian noise. This does not reflect real attacks, which manifest as structured, sparse perturbations in specific feature subsets:

| Attack Type | Features Perturbed | Mechanism |
|---|---|---|
| **SYN Flood** | `SYN Flag Count`, `Flow Packets/s`, `Flow Duration` | High packet rate, tiny duration, many SYN flags |
| **Port Scan** | `Destination Port`-related, `RST Flag Count`, `Flow Duration` | Many short flows, many RSTs, varying ports |
| **DDoS Volume** | `Flow Bytes/s`, `Flow Packets/s`, `Total Fwd Packets` | Massive throughput burst |
| **Data Exfiltration** | `Bwd Packet Length Mean`, `Flow Bytes/s`, `ACK Flag Count` | Large backward packets, sustained ACK-heavy flow |

This produces anomalies that are statistically discriminative in *exactly the features the model should focus on*, yielding metrics that reflect realistic detection capability.


In [ ]:
# ============================================================
# SECTION 1.3 : REALISTIC ANOMALY SIMULATION & SPLITS
# ============================================================
# FIX 1 — Attack-type-specific structured perturbations
# instead of uniform Gaussian noise across all 53 features.
#
# Split strategy (no threshold leakage):
#   80% train   → model training (100% benign)
#   10% val     → threshold calibration for Autoencoder
#   10% test    → final held-out evaluation (never touched during training)
# ============================================================

X_all = df.drop(columns=['Label']).values.astype(np.float32)
feature_names = df.drop(columns=['Label']).columns.tolist()
y_all = np.zeros(len(X_all), dtype=int)   # all benign initially

# ── Step 1 : Three-way split BEFORE anomaly injection ─────────────────────
# Stratify on y_all (all 0s), so this is just random splitting.
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f'Split sizes (before anomaly injection):')
print(f'  X_train : {X_train.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')

# ── Step 2 : Attack-type-specific perturbation function ───────────────────
def get_feature_idx(names, keywords):
    """Return indices of features whose name contains any keyword (case-insensitive)."""
    return [i for i, n in enumerate(names)
            if any(kw.lower() in n.lower() for kw in keywords)]

# Define attack profiles — (keywords to target, multiplier on feature std)
ATTACK_PROFILES = {
    'SYN Flood': {
        'keywords': ['SYN Flag', 'Packets/s', 'Flow Duration'],
        'multiplier': 8.0,
    },
    'Port Scan': {
        'keywords': ['RST Flag', 'Flow Duration', 'Port_'],
        'multiplier': 6.0,
    },
    'DDoS Volume': {
        'keywords': ['Bytes/s', 'Packets/s', 'Total Fwd Packets'],
        'multiplier': 10.0,
    },
    'Data Exfiltration': {
        'keywords': ['Bwd Packet Length', 'Bytes/s', 'ACK Flag'],
        'multiplier': 5.0,
    },
}

def inject_structured_anomalies(X, y, feature_names, anomaly_fraction=0.15, seed=42):
    """
    Inject structured, attack-type-specific anomalies into a copy of X.
    Each anomaly is assigned one attack profile; only the relevant feature
    subset is perturbed (high multiplier × feature std).
    Returns (X_perturbed, y_perturbed).
    """
    rng = np.random.RandomState(seed)
    X_out = X.copy()
    y_out = y.copy()

    n_anomalies = int(anomaly_fraction * len(X))
    anomaly_idx = rng.choice(len(X), n_anomalies, replace=False)

    # Pre-compute std per feature (from the array itself)
    feat_std = X.std(axis=0)

    # Assign each anomaly a random attack profile
    profile_names = list(ATTACK_PROFILES.keys())
    assigned_profiles = rng.choice(len(profile_names), n_anomalies)

    for i, (row_idx, profile_id) in enumerate(zip(anomaly_idx, assigned_profiles)):
        profile = ATTACK_PROFILES[profile_names[profile_id]]
        feat_idx = get_feature_idx(feature_names, profile['keywords'])
        if not feat_idx:
            # Fallback if no features match: perturb a random subset
            feat_idx = rng.choice(len(feature_names), max(3, len(feature_names)//10),
                                  replace=False).tolist()
        mult = profile['multiplier']
        noise = rng.normal(0, mult, len(feat_idx)).astype(np.float32)
        X_out[row_idx, feat_idx] += noise * feat_std[feat_idx]
        y_out[row_idx] = 1

    return X_out, y_out

# ── Step 3 : Inject anomalies into val and test sets only ─────────────────
X_val_aug,  y_val_aug  = inject_structured_anomalies(
    X_val,  y_val,  feature_names, anomaly_fraction=0.15, seed=SEED)
X_test_aug, y_test_aug = inject_structured_anomalies(
    X_test, y_test, feature_names, anomaly_fraction=0.15, seed=SEED+1)

# Training set stays 100% benign (one-class learning)
X_train_normal = X_train[y_train == 0]

print()
print('After structured anomaly injection:')
print(f'  X_train_normal : {X_train_normal.shape}  (100% benign — model training)')
print(f'  X_val_aug      : {X_val_aug.shape}   '
      f'({(y_val_aug==0).sum()} benign | {(y_val_aug==1).sum()} anomalies) ← threshold calibration')
print(f'  X_test_aug     : {X_test_aug.shape}   '
      f'({(y_test_aug==0).sum()} benign | {(y_test_aug==1).sum()} anomalies) ← final evaluation')
print()
print('Attack profiles used:')
for name, prof in ATTACK_PROFILES.items():
    idxs = get_feature_idx(feature_names, prof['keywords'])
    print(f'  {name:<22} → {len(idxs)} features perturbed  '
          f'(mult={prof["multiplier"]}σ)')
print()
print('⚠  NOTE: Anomalies are structured (attack-type-specific).')
print('   Only relevant feature subsets are perturbed per attack type.')
print('   Metrics now reflect realistic detection capability.')


**Split summary:**
- `X_train_normal` — 100% benign, used for one-class model training.
- `X_val_aug` — benign + structured anomalies, used **only** for Autoencoder threshold calibration and cross-validation scoring. **Never used to report final metrics.**
- `X_test_aug` — benign + structured anomalies, used **only** for final held-out evaluation. **Threshold is fixed before this set is touched.**


---
## Section 2 — Model Selection Justification

### Why Unsupervised Anomaly Detection?

| Criterion | Implication |
|---|---|
| **Extreme class imbalance** | Supervised classifiers bias towards the majority (benign) class; F1-score, AUC-ROC, and PR-AUC are the correct metrics |
| **Zero-day attack generalisation** | One-class models generalise by learning *what is normal*, flagging any deviation — including novel attack patterns not seen during training |
| **Label scarcity** | Only benign traffic labels are available; unsupervised models require no attack labels |

### Autoencoder — Principle
A neural network trained to reconstruct its input through a compressed bottleneck. Reconstruction error (MSE) is low for benign flows and high for anomalous flows the network has not learned to reconstruct.

### Isolation Forest — Principle
Ensemble of random isolation trees. Anomalous points require fewer partitions to isolate → shorter average path length → higher anomaly score.


---
## Section 3 — Model 1: Autoencoder

### 3.1 Architecture


In [ ]:
# ============================================================
# SECTION 3.1 : AUTOENCODER ARCHITECTURE
# ============================================================
INPUT_DIM = X_train_normal.shape[1]

def build_autoencoder(input_dim, encoding_dim=8):
    """
    Symmetric encoder-decoder autoencoder.
    input_dim    : number of input features
    encoding_dim : bottleneck dimension
    Returns (autoencoder, encoder) Keras models.
    """
    inp = Input(shape=(input_dim,), name='Input')

    # Encoder
    x = Dense(32, activation='relu', name='Enc_32')(inp)
    x = BatchNormalization(name='BN_enc_32')(x)
    x = Dropout(0.2, name='Drop_enc_32')(x)

    x = Dense(16, activation='relu', name='Enc_16')(x)
    x = BatchNormalization(name='BN_enc_16')(x)
    x = Dropout(0.2, name='Drop_enc_16')(x)

    bottleneck = Dense(encoding_dim, activation='relu', name='Bottleneck')(x)

    # Decoder
    x = Dense(16, activation='relu', name='Dec_16')(bottleneck)
    x = BatchNormalization(name='BN_dec_16')(x)
    x = Dropout(0.2, name='Drop_dec_16')(x)

    x = Dense(32, activation='relu', name='Dec_32')(x)
    x = BatchNormalization(name='BN_dec_32')(x)
    x = Dropout(0.2, name='Drop_dec_32')(x)

    output = Dense(input_dim, activation='linear', name='Output')(x)

    autoencoder = Model(inputs=inp, outputs=output, name='Autoencoder')
    encoder     = Model(inputs=inp, outputs=bottleneck, name='Encoder')
    return autoencoder, encoder


autoencoder, encoder = build_autoencoder(INPUT_DIM, encoding_dim=8)
autoencoder.compile(optimizer=Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
autoencoder.summary()


### 3.2 Training (on benign-only training set)


In [ ]:
# ============================================================
# SECTION 3.2 : AUTOENCODER TRAINING
# ============================================================
# Trained ONLY on benign flows (X_train_normal).
# The validation split here is a 15% slice of X_train_normal —
# NOT the val_aug set — so the threshold calibration set remains
# completely unseen during training.
# ============================================================

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1),
]

history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=100,
    batch_size=256,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1,
)

print(f'\n✓ Training complete — {len(history.history["loss"])} effective epochs')


### 3.3 Learning Curves


In [ ]:
# ============================================================
# SECTION 3.3 : LEARNING CURVES
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss (MSE)', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss (MSE)',   linewidth=2, linestyle='--')
axes[0].set_title('Loss Curve — Autoencoder', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].legend()

axes[1].plot(history.history['mae'],     label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE',   linewidth=2, linestyle='--')
axes[1].set_title('MAE Curve — Autoencoder', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE'); axes[1].legend()

plt.suptitle('Autoencoder Learning Curves (trained on benign flows only)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ae_learning_curves.png', bbox_inches='tight')
plt.show()

print(f'  Final train loss : {history.history["loss"][-1]:.6f}')
print(f'  Final val loss   : {history.history["val_loss"][-1]:.6f}')


### 3.4 Reconstruction Error Analysis


In [ ]:
# ============================================================
# SECTION 3.4 : RECONSTRUCTION ERRORS — VALIDATION SET
# (used ONLY for threshold calibration, not for reporting metrics)
# ============================================================

# Compute reconstruction errors on the validation set
X_val_recon = autoencoder.predict(X_val_aug, verbose=0)
ae_val_errors = np.mean(np.square(X_val_aug - X_val_recon), axis=1)

print('Reconstruction errors on VALIDATION set:')
print(f'  Benign  — mean MSE : {ae_val_errors[y_val_aug==0].mean():.6f}')
print(f'  Anomaly — mean MSE : {ae_val_errors[y_val_aug==1].mean():.6f}')
ratio = ae_val_errors[y_val_aug==1].mean() / (ae_val_errors[y_val_aug==0].mean() + 1e-10)
print(f'  Ratio (anomaly/benign) : {ratio:.2f}x  '
      f'(higher = better separation)')


### 3.5 Distribution of Reconstruction Errors (Validation Set)


In [ ]:
# ============================================================
# SECTION 3.5 : ERROR DISTRIBUTION VISUALISATION (VAL SET)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(ae_val_errors[y_val_aug==0], bins=80, alpha=0.6,
             color='#2ecc71', label='Benign', density=True)
axes[0].hist(ae_val_errors[y_val_aug==1], bins=80, alpha=0.6,
             color='#e74c3c', label='Anomaly', density=True)
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Density')
axes[0].set_title('Reconstruction Error Distribution\n(Benign vs Anomaly — Val Set)',
                  fontweight='bold')
axes[0].legend()
axes[0].set_xlim([0, np.percentile(ae_val_errors, 99)])

labels_str = np.where(y_val_aug == 0, 'Benign', 'Anomaly')
df_plot = pd.DataFrame({'MSE': ae_val_errors, 'Class': labels_str})
df_plot.boxplot(column='MSE', by='Class', ax=axes[1],
                boxprops=dict(color='navy'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Boxplot — Reconstruction Errors by Class (Val)', fontweight='bold')
axes[1].set_xlabel('Class'); axes[1].set_ylabel('MSE')

plt.suptitle('Autoencoder: Benign / Anomaly Separation via Reconstruction Error (Val Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ae_reconstruction_errors.png', bbox_inches='tight')
plt.show()


### 3.6 Threshold Calibration on Validation Set (FIX: No Test Leakage)

**Critical fix applied:** The optimal threshold is now selected using the **held-out validation set** (`X_val_aug`), which is completely disjoint from the test set. The test set is never examined during threshold selection. Only after the threshold is fixed does the test set evaluation proceed.


In [ ]:
# ============================================================
# SECTION 3.6 : THRESHOLD CALIBRATION — VALIDATION SET ONLY
# FIX: threshold is chosen on val set → test set is never touched here
# ============================================================
from sklearn.metrics import precision_recall_curve

precisions_val, recalls_val, thresholds_val = precision_recall_curve(
    y_val_aug, ae_val_errors
)

# F1-optimal threshold on validation set
f1_scores_val = (2 * precisions_val[:-1] * recalls_val[:-1]
                 / (precisions_val[:-1] + recalls_val[:-1] + 1e-8))
best_idx      = np.argmax(f1_scores_val)
AE_THRESHOLD  = thresholds_val[best_idx]

print('=' * 60)
print('AUTOENCODER THRESHOLD (calibrated on VALIDATION set)')
print('=' * 60)
print(f'  Threshold MSE     : {AE_THRESHOLD:.6f}')
print(f'  Val F1 at threshold: {f1_scores_val[best_idx]:.4f}')
print(f'  Val Precision     : {precisions_val[best_idx]:.4f}')
print(f'  Val Recall        : {recalls_val[best_idx]:.4f}')
print()
print('  ✓ Threshold is now FIXED.')
print('  ✓ Test set will be evaluated with this fixed threshold.')
print('  ✓ No information from the test set was used.')

# Visualise threshold on PR curve (val set)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recalls_val[:-1], precisions_val[:-1],
        color='#3498db', linewidth=2, label='PR curve (val)')
ax.axvline(recalls_val[best_idx], color='red', linestyle='--', alpha=0.7)
ax.axhline(precisions_val[best_idx], color='red', linestyle='--', alpha=0.7,
           label=f'F1-optimal threshold (F1={f1_scores_val[best_idx]:.3f})')
ax.scatter([recalls_val[best_idx]], [precisions_val[best_idx]],
           color='red', s=100, zorder=5)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curve — Autoencoder (Validation Set)\nThreshold selected at F1 maximum',
             fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('ae_threshold_calibration.png', bbox_inches='tight')
plt.show()


### 3.7 Final Predictions on Test Set


In [ ]:
# ============================================================
# SECTION 3.7 : AUTOENCODER — TEST SET EVALUATION
# Threshold is already fixed from Section 3.6 (val set only)
# ============================================================

X_test_recon  = autoencoder.predict(X_test_aug, verbose=0)
ae_test_errors = np.mean(np.square(X_test_aug - X_test_recon), axis=1)

# Apply fixed threshold
y_pred_ae = (ae_test_errors > AE_THRESHOLD).astype(int)

print('Reconstruction errors on TEST set:')
print(f'  Benign  — mean MSE : {ae_test_errors[y_test_aug==0].mean():.6f}')
print(f'  Anomaly — mean MSE : {ae_test_errors[y_test_aug==1].mean():.6f}')
print(f'  Fixed threshold    : {AE_THRESHOLD:.6f}')
print()
print(f'  Predicted benign  : {(y_pred_ae==0).sum():,}')
print(f'  Predicted anomaly : {(y_pred_ae==1).sum():,}')
print(f'  True benign       : {(y_test_aug==0).sum():,}')
print(f'  True anomaly      : {(y_test_aug==1).sum():,}')


---
## Section 4 — Model 2: Isolation Forest

### 4.1 Training with Correct Contamination Strategy

**Fix applied:** The original code attempted to estimate the contamination rate from `y_train`, which is 100% benign (all zeros), so `contamination_rate` was always 0 and the fallback to `'auto'` always triggered — making the estimation logic dead code. The fix:
- When the training set is 100% benign (the correct one-class setup), `contamination='auto'` is set explicitly and intentionally.
- The effective anomaly rate for interpretive purposes is estimated from `y_val_aug` (the labelled validation set), reported transparently.


In [ ]:
# ============================================================
# SECTION 4.1 : ISOLATION FOREST — TRAINING
# FIX: contamination='auto' is correct and intentional for
#      one-class training. Contamination from val set is
#      reported for transparency only.
# ============================================================

# Training set is 100% benign by design — correct for one-class IF
assert y_train.sum() == 0, 'Training set should be 100% benign'

# Estimated anomaly prevalence from validation set (informational only)
val_anomaly_rate = y_val_aug.mean()
print(f'Contamination strategy:')
print(f'  Training set is 100% benign → contamination="auto" (correct & intentional)')
print(f'  Estimated anomaly prevalence (from val set): {val_anomaly_rate:.3f} '
      f'({val_anomaly_rate*100:.1f}%)')
print(f'  Note: "auto" sets the threshold at the iForest score for the expected')
print(f'        inlier boundary (~10% contamination equivalent internally).')
print()

iso_forest = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination='auto',   # FIX: explicit, intentional — not a fallback
    max_features=1.0,
    bootstrap=False,
    random_state=SEED,
    n_jobs=-1,
    verbose=0,
)

iso_forest.fit(X_train)   # full training set (all benign)

print('✓ Isolation Forest trained')
print(f'  n_estimators : {iso_forest.n_estimators}')
print(f'  max_samples  : {iso_forest.max_samples_}')
print(f'  contamination: auto')


### 4.2 Predictions on Test Set


In [ ]:
# ============================================================
# SECTION 4.2 : ISOLATION FOREST — TEST PREDICTIONS
# ============================================================
# Convention: IF returns +1 (normal) / -1 (anomaly)
# Convert to: 0 (benign) / 1 (anomaly) for consistency

if_raw = iso_forest.predict(X_test_aug)
y_pred_if = np.where(if_raw == 1, 0, 1)

# Anomaly score: higher = more anomalous (negated decision_function)
if_anomaly_scores = -iso_forest.decision_function(X_test_aug)

# Normalise to [0,1] for calibrated PR/ROC comparison with AE scores
scaler_scores = MinMaxScaler()
if_anomaly_scores_norm = scaler_scores.fit_transform(
    if_anomaly_scores.reshape(-1, 1)).ravel()
ae_test_errors_norm = MinMaxScaler().fit_transform(
    ae_test_errors.reshape(-1, 1)).ravel()

print('Isolation Forest predictions on test set:')
print(f'  Predicted benign  : {(y_pred_if==0).sum():,}')
print(f'  Predicted anomaly : {(y_pred_if==1).sum():,}')
print(f'  True benign       : {(y_test_aug==0).sum():,}')
print(f'  True anomaly      : {(y_test_aug==1).sum():,}')


### 4.3 Isolation Forest Score Distribution


In [ ]:
# ============================================================
# SECTION 4.3 : IF SCORE DISTRIBUTION VISUALISATION
# ============================================================
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(if_anomaly_scores[y_test_aug==0], bins=80, alpha=0.6,
        color='#2ecc71', label='Benign', density=True)
ax.hist(if_anomaly_scores[y_test_aug==1], bins=80, alpha=0.6,
        color='#e74c3c', label='Anomaly', density=True)
ax.set_xlabel('Isolation Forest Anomaly Score (higher = more anomalous)')
ax.set_ylabel('Density')
ax.set_title('Isolation Forest: Anomaly Score Distribution (Test Set)',
             fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('if_anomaly_scores.png', bbox_inches='tight')
plt.show()


---
## Section 5 — Cross-Validation Stability Analysis

**Fix applied:** Both models are now evaluated with 5-fold StratifiedKFold cross-validation to estimate metric variance and confirm that results are not artefacts of a single train/test split.

**Protocol:**
- Each fold: train on 80% benign-only, inject anomalies into the 20% held-out fold, evaluate.
- The Autoencoder threshold is re-calibrated on a 10% slice of each fold's held-out set (inner val), and the remaining 10% is used for metric computation.
- Reported: mean ± std of F1, Recall, Precision, AUC-ROC across 5 folds.


In [ ]:
# ============================================================
# SECTION 5 : 5-FOLD CROSS-VALIDATION STABILITY
# ============================================================
# For the Autoencoder, full training per fold is expensive.
# We use a lighter version (50 epochs max) to keep runtime tractable
# while still estimating variance. Isolation Forest is fast — full run.
# ============================================================

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# We run CV on X_all with synthetic y labels (0=benign) — anomalies are injected per fold
# Combine train+val+test back into a benign pool for CV
X_benign_pool = X_all.copy()
y_benign_pool = np.zeros(len(X_benign_pool), dtype=int)  # all benign before injection

cv_results = {'fold': [], 'model': [],
              'F1': [], 'Recall': [], 'Precision': [], 'AUC_ROC': []}

print('=' * 65)
print(f'5-FOLD CROSS-VALIDATION  (structured anomaly injection per fold)')
print('=' * 65)

for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(X_benign_pool, y_benign_pool)):

    fold_num = fold_idx + 1
    X_tr = X_benign_pool[train_idx]
    X_ho = X_benign_pool[test_idx]
    y_ho_clean = y_benign_pool[test_idx]

    # Split held-out into inner-val (threshold) + inner-test (metrics)
    X_ho_val, X_ho_test, y_ho_val_c, y_ho_test_c = train_test_split(
        X_ho, y_ho_clean, test_size=0.50, random_state=SEED
    )

    # Inject anomalies into both inner splits
    X_ho_val,  y_ho_val  = inject_structured_anomalies(
        X_ho_val,  y_ho_val_c,  feature_names, 0.15, seed=fold_idx)
    X_ho_test, y_ho_test = inject_structured_anomalies(
        X_ho_test, y_ho_test_c, feature_names, 0.15, seed=fold_idx+100)

    # ── Isolation Forest ──────────────────────────────────────────────────
    if_fold = IsolationForest(n_estimators=100, contamination='auto',
                              random_state=SEED, n_jobs=-1)
    if_fold.fit(X_tr)
    if_raw_fold = if_fold.predict(X_ho_test)
    y_pred_if_fold = np.where(if_raw_fold == 1, 0, 1)
    if_scores_fold = -if_fold.decision_function(X_ho_test)

    # Guard: skip fold if only one class in y_ho_test
    if len(np.unique(y_ho_test)) < 2:
        continue

    cv_results['fold'].append(fold_num)
    cv_results['model'].append('Isolation Forest')
    cv_results['F1'].append(f1_score(y_ho_test, y_pred_if_fold, zero_division=0))
    cv_results['Recall'].append(recall_score(y_ho_test, y_pred_if_fold, zero_division=0))
    cv_results['Precision'].append(precision_score(y_ho_test, y_pred_if_fold, zero_division=0))
    cv_results['AUC_ROC'].append(roc_auc_score(y_ho_test, if_scores_fold))

    # ── Autoencoder (lightweight per-fold training) ───────────────────────
    ae_fold, _ = build_autoencoder(X_tr.shape[1], encoding_dim=8)
    ae_fold.compile(optimizer=Adam(1e-3), loss='mse')
    ae_fold.fit(
        X_tr, X_tr,
        epochs=50,
        batch_size=256,
        validation_split=0.10,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
        verbose=0,
    )

    # Calibrate threshold on inner val set
    val_recon  = ae_fold.predict(X_ho_val, verbose=0)
    val_errors = np.mean(np.square(X_ho_val - val_recon), axis=1)
    prec_v, rec_v, thr_v = precision_recall_curve(y_ho_val, val_errors)
    f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    fold_threshold = thr_v[np.argmax(f1_v)]

    # Evaluate on inner test set
    test_recon  = ae_fold.predict(X_ho_test, verbose=0)
    test_errors = np.mean(np.square(X_ho_test - test_recon), axis=1)
    y_pred_ae_fold = (test_errors > fold_threshold).astype(int)

    cv_results['fold'].append(fold_num)
    cv_results['model'].append('Autoencoder')
    cv_results['F1'].append(f1_score(y_ho_test, y_pred_ae_fold, zero_division=0))
    cv_results['Recall'].append(recall_score(y_ho_test, y_pred_ae_fold, zero_division=0))
    cv_results['Precision'].append(precision_score(y_ho_test, y_pred_ae_fold, zero_division=0))
    cv_results['AUC_ROC'].append(roc_auc_score(y_ho_test, test_errors))

    print(f'  Fold {fold_num}/5 complete')

cv_df = pd.DataFrame(cv_results)
print()
print('Cross-Validation Results (mean ± std across 5 folds):')
print('=' * 65)
for model_name in ['Autoencoder', 'Isolation Forest']:
    sub = cv_df[cv_df['model'] == model_name]
    print(f'\n  {model_name}')
    for metric in ['F1', 'Recall', 'Precision', 'AUC_ROC']:
        vals = sub[metric]
        print(f'    {metric:<12}: {vals.mean():.4f} ± {vals.std():.4f}  '
              f'[{vals.min():.4f} – {vals.max():.4f}]')


### 5.1 Cross-Validation Stability Plot


In [ ]:
# ============================================================
# SECTION 5.1 : CV STABILITY VISUALISATION
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_cv = ['F1', 'Recall', 'Precision', 'AUC_ROC']
colors_cv  = {'Autoencoder': '#3498db', 'Isolation Forest': '#e67e22'}

for ax, metric in zip(axes.flatten(), metrics_cv):
    for model_name, color in colors_cv.items():
        sub = cv_df[cv_df['model'] == model_name]
        folds = sub['fold'].values
        vals  = sub[metric].values
        ax.plot(folds, vals, marker='o', linewidth=2,
                color=color, label=model_name, alpha=0.8)
        ax.axhline(vals.mean(), color=color, linestyle='--',
                   alpha=0.5, linewidth=1.5)
        ax.fill_between(folds,
                        vals.mean() - vals.std(),
                        vals.mean() + vals.std(),
                        color=color, alpha=0.1)
    ax.set_title(f'{metric} — Stability across Folds', fontweight='bold')
    ax.set_xlabel('Fold'); ax.set_ylabel(metric)
    ax.set_xticks(range(1, N_FOLDS + 1))
    ax.legend(); ax.grid(True, alpha=0.4)
    ax.set_ylim([0, 1.05])

plt.suptitle('5-Fold Cross-Validation Stability\n(Autoencoder vs Isolation Forest)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cv_stability.png', bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print('  - Dashed lines = fold mean; shaded area = ±1 std')
print('  - Low std across folds → results are stable, not split-dependent')
print('  - High std → model is sensitive to the specific data split')


---
## Section 6 — Full Evaluation & Comparative Analysis

### 6.1 Evaluation Function


In [ ]:
# ============================================================
# SECTION 6.1 : EVALUATION FUNCTION
# ============================================================
def evaluate_model(model_name, y_true, y_pred, anomaly_scores):
    """
    Full evaluation suite: accuracy, precision, recall, F1,
    AUC-ROC, PR-AUC, confusion matrix breakdown.
    """
    print('=' * 60)
    print(f'EVALUATION : {model_name}')
    print('=' * 60)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc_roc = roc_auc_score(y_true, anomaly_scores)
    pr_auc  = average_precision_score(y_true, anomaly_scores)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}  ← primary metric (missed attacks = FN)')
    print(f'  F1-score  : {f1:.4f}')
    print(f'  AUC-ROC   : {auc_roc:.4f}')
    print(f'  PR-AUC    : {pr_auc:.4f}')
    print()
    print('  Confusion matrix:')
    print(f'    TP (attacks detected) : {tp:,}')
    print(f'    TN (benign correct)   : {tn:,}')
    print(f'    FP (false alarms)     : {fp:,}')
    print(f'    FN (attacks missed)   : {fn:,}')
    print()
    print(classification_report(y_true, y_pred,
                                 target_names=['Benign', 'Anomaly']))
    return {
        'Model': model_name, 'Accuracy': round(acc,4),
        'Precision': round(prec,4), 'Recall': round(rec,4),
        'F1-score': round(f1,4), 'AUC-ROC': round(auc_roc,4),
        'PR-AUC': round(pr_auc,4),
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
    }

metrics_ae = evaluate_model(
    'Autoencoder', y_test_aug, y_pred_ae, ae_test_errors_norm)
metrics_if = evaluate_model(
    'Isolation Forest', y_test_aug, y_pred_if, if_anomaly_scores_norm)


### 6.2 Confusion Matrices


In [ ]:
# ============================================================
# SECTION 6.2 : CONFUSION MATRICES
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model_name, y_pred) in zip(
    axes,
    [('Autoencoder', y_pred_ae), ('Isolation Forest', y_pred_if)]
):
    cm = confusion_matrix(y_test_aug, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benign (pred)', 'Anomaly (pred)'],
                yticklabels=['Benign (true)', 'Anomaly (true)'],
                ax=ax, linewidths=0.5, linecolor='gray',
                annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f'Confusion Matrix\n{model_name}',
                 fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices — Autoencoder vs Isolation Forest',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()


### 6.3 ROC Curves


In [ ]:
# ============================================================
# SECTION 6.3 : ROC CURVES
# ============================================================
fig, ax = plt.subplots(figsize=(8, 6))

fpr_ae, tpr_ae, _ = roc_curve(y_test_aug, ae_test_errors_norm)
auc_ae = roc_auc_score(y_test_aug, ae_test_errors_norm)
ax.plot(fpr_ae, tpr_ae, linewidth=2, color='#3498db',
        label=f'Autoencoder (AUC = {auc_ae:.4f})')

fpr_if, tpr_if, _ = roc_curve(y_test_aug, if_anomaly_scores_norm)
auc_if = roc_auc_score(y_test_aug, if_anomaly_scores_norm)
ax.plot(fpr_if, tpr_if, linewidth=2, color='#e67e22',
        label=f'Isolation Forest (AUC = {auc_if:.4f})')

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=11)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=11)
ax.set_title('ROC Curves — Autoencoder vs Isolation Forest', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.4)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()


### 6.4 Precision-Recall Curves


In [ ]:
# ============================================================
# SECTION 6.4 : PRECISION-RECALL CURVES
# ============================================================
fig, ax = plt.subplots(figsize=(8, 6))

prec_ae, rec_ae, _ = precision_recall_curve(y_test_aug, ae_test_errors_norm)
prauc_ae = average_precision_score(y_test_aug, ae_test_errors_norm)
ax.plot(rec_ae, prec_ae, linewidth=2, color='#3498db',
        label=f'Autoencoder (PR-AUC = {prauc_ae:.4f})')

prec_if, rec_if, _ = precision_recall_curve(y_test_aug, if_anomaly_scores_norm)
prauc_if = average_precision_score(y_test_aug, if_anomaly_scores_norm)
ax.plot(rec_if, prec_if, linewidth=2, color='#e67e22',
        label=f'Isolation Forest (PR-AUC = {prauc_if:.4f})')

baseline = y_test_aug.mean()
ax.axhline(baseline, color='gray', linestyle='--', linewidth=1,
           label=f'Baseline (anomaly rate = {baseline:.2f})')

ax.set_xlabel('Recall', fontsize=11); ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves — Autoencoder vs Isolation Forest',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.4)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig('pr_curves.png', bbox_inches='tight')
plt.show()


### 6.5 Comparative Metrics Table & Heatmap


In [ ]:
# ============================================================
# SECTION 6.5 : COMPARATIVE METRICS TABLE & HEATMAP
# ============================================================
metric_keys = ['Model','Accuracy','Precision','Recall','F1-score','AUC-ROC','PR-AUC']
df_metrics = pd.DataFrame([
    {k: metrics_ae[k] for k in metric_keys},
    {k: metrics_if[k] for k in metric_keys},
]).set_index('Model').fillna(0.0)

print('=' * 65)
print('COMPARATIVE METRICS TABLE (held-out test set)')
print('=' * 65)
print(df_metrics.round(4).to_string())
print()
print('Best model per metric:')
for col in df_metrics.columns:
    best = df_metrics[col].idxmax()
    print(f'  {col:<12} → {best}  ({df_metrics.loc[best, col]:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metric_cols = ['Accuracy','Precision','Recall','F1-score','AUC-ROC','PR-AUC']
x = np.arange(len(metric_cols)); width = 0.35

bars1 = axes[0].bar(x - width/2,
                    [metrics_ae[m] for m in metric_cols],
                    width, label='Autoencoder', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = axes[0].bar(x + width/2,
                    [metrics_if[m] for m in metric_cols],
                    width, label='Isolation Forest', color='#e67e22', alpha=0.8, edgecolor='black')
for bar in list(bars1) + list(bars2):
    axes[0].annotate(f'{bar.get_height():.3f}',
                     xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=8, rotation=45)
axes[0].set_xticks(x); axes[0].set_xticklabels(metric_cols, rotation=20, ha='right')
axes[0].set_ylim([0, 1.15]); axes[0].set_ylabel('Score')
axes[0].set_title('Metric Comparison', fontweight='bold', fontsize=12)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.4)

sns.heatmap(df_metrics, annot=True, fmt='.4f', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, linecolor='gray', ax=axes[1],
            annot_kws={'size': 12, 'weight': 'bold'})
axes[1].set_title('Metrics Heatmap\n(green = better)', fontweight='bold', fontsize=12)

plt.suptitle('Autoencoder vs Isolation Forest — AIoT-6G (Held-Out Test Set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('metrics_comparison.png', bbox_inches='tight')
plt.show()
df_metrics.to_csv('metrics_comparison.csv')
print('→ metrics_comparison.csv saved')


### 6.6 Autoencoder Latent Space (PCA Projection)


In [ ]:
# ============================================================
# SECTION 6.6 : AUTOENCODER LATENT SPACE ANALYSIS
# ============================================================
X_test_encoded = encoder.predict(X_test_aug, verbose=0)
pca = PCA(n_components=2, random_state=SEED)
X_test_2d = pca.fit_transform(X_test_encoded)

print(f'PCA variance explained (2D): {pca.explained_variance_ratio_.sum()*100:.1f}%')

fig, ax = plt.subplots(figsize=(10, 7))
colors_map = {0: '#2ecc71', 1: '#e74c3c'}
labels_map  = {0: 'Benign', 1: 'Anomaly'}
for cls in [0, 1]:
    mask = y_test_aug == cls
    ax.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
               c=colors_map[cls], label=labels_map[cls],
               alpha=0.4, s=10, edgecolors='none')
ax.set_title('Latent Space (Bottleneck 8D → PCA 2D)\nBenign / Anomaly Separation',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(fontsize=11, markerscale=3); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('latent_space_pca.png', bbox_inches='tight')
plt.show()


### 6.7 Feature Importance — Isolation Forest


In [ ]:
# ============================================================
# SECTION 6.7 : FEATURE IMPORTANCE (ISOLATION FOREST)
# Note: correlation with anomaly score is an approximation.
# Because anomalies are structured (attack-specific), this now
# reflects genuine discriminative features, not just high-std features.
# ============================================================
feature_names_list = df.drop(columns=['Label']).columns.tolist()
correlations = np.abs(np.corrcoef(X_test_aug.T, if_anomaly_scores)[-1, :-1])
feat_imp_df = pd.DataFrame({
    'Feature': feature_names_list,
    'Abs Correlation with Anomaly Score': correlations
}).sort_values('Abs Correlation with Anomaly Score', ascending=False)

print('Top 15 features most correlated with IF anomaly score:')
print(feat_imp_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
top15 = feat_imp_df.head(15)
ax.barh(top15['Feature'][::-1],
        top15['Abs Correlation with Anomaly Score'][::-1],
        color='#e67e22', alpha=0.8, edgecolor='black')
ax.set_xlabel('|r| with anomaly score')
ax.set_title('Top 15 Features — Isolation Forest Importance Proxy',
             fontweight='bold', fontsize=12)
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('if_feature_importance.png', bbox_inches='tight')
plt.show()


---
## Section 7 — Model Persistence

All trained artefacts are saved to the `saved_models/` directory:

| File | Format | Contents |
|---|---|---|
| `isolation_forest.pkl` | pickle | Trained IsolationForest object |
| `autoencoder.keras` | Keras native | Full autoencoder (encoder + decoder weights) |
| `encoder.keras` | Keras native | Encoder sub-model (bottleneck representations) |
| `ae_threshold.pkl` | pickle | Calibrated MSE threshold scalar |
| `scaler_scores.pkl` | pickle | MinMaxScaler for IF score normalisation |
| `cv_results.csv` | CSV | Cross-validation fold-level results |

**Why `.keras` over `.h5`:** The `.keras` format is the Keras 3 native format, recommended since TF ≥ 2.12. The legacy `.h5` format does not support custom objects cleanly and is deprecated.


In [ ]:
# ============================================================
# SECTION 7 : MODEL PERSISTENCE — SAVE ALL ARTEFACTS
# ============================================================

# ── 1. Isolation Forest → pickle ──────────────────────────────────────────
if_path = os.path.join(MODELS_DIR, 'isolation_forest.pkl')
with open(if_path, 'wb') as f:
    pickle.dump(iso_forest, f)
print(f'✓ Isolation Forest saved      → {if_path}')

# ── 2. Autoencoder (full model) → Keras native ────────────────────────────
ae_path = os.path.join(MODELS_DIR, 'autoencoder.keras')
autoencoder.save(ae_path)
print(f'✓ Autoencoder saved           → {ae_path}')

# ── 3. Encoder sub-model → Keras native ───────────────────────────────────
enc_path = os.path.join(MODELS_DIR, 'encoder.keras')
encoder.save(enc_path)
print(f'✓ Encoder saved               → {enc_path}')

# ── 4. AE threshold → pickle ──────────────────────────────────────────────
threshold_path = os.path.join(MODELS_DIR, 'ae_threshold.pkl')
with open(threshold_path, 'wb') as f:
    pickle.dump(float(AE_THRESHOLD), f)
print(f'✓ AE threshold saved          → {threshold_path}  '
      f'(value: {AE_THRESHOLD:.6f})')

# ── 5. Score normalisation scaler → pickle ────────────────────────────────
scaler_path = os.path.join(MODELS_DIR, 'scaler_scores.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler_scores, f)
print(f'✓ Score MinMaxScaler saved    → {scaler_path}')

# ── 6. CV results → CSV ───────────────────────────────────────────────────
cv_csv_path = os.path.join(MODELS_DIR, 'cv_results.csv')
cv_df.to_csv(cv_csv_path, index=False)
print(f'✓ CV results saved            → {cv_csv_path}')

# ── Reload verification ───────────────────────────────────────────────────
print()
print('--- Reload verification ---')

with open(if_path, 'rb') as f:
    iso_forest_loaded = pickle.load(f)
test_pred_check = iso_forest_loaded.predict(X_test_aug[:5])
print(f'  IF reload OK  — n_estimators={iso_forest_loaded.n_estimators}, '
      f'sample predict: {test_pred_check}')

ae_loaded = load_model(ae_path)
recon_check = ae_loaded.predict(X_test_aug[:3], verbose=0)
print(f'  AE reload OK  — input shape: {ae_loaded.input_shape}, '
      f'output shape: {recon_check.shape}')

with open(threshold_path, 'rb') as f:
    thr_loaded = pickle.load(f)
print(f'  Threshold OK  — value: {thr_loaded:.6f}')

with open(scaler_path, 'rb') as f:
    scaler_loaded = pickle.load(f)
print(f'  Scaler OK     — feature range: {scaler_loaded.feature_range}')

print()
print(f'✓ All artefacts saved and verified in: {MODELS_DIR}/')
print()
print('  File inventory:')
for fname in sorted(os.listdir(MODELS_DIR)):
    fpath = os.path.join(MODELS_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'    {fname:<35}  {size_kb:>8.1f} KB')


**Reuse pattern for inference:**

```python
import pickle, numpy as np
from tensorflow.keras.models import load_model

# Load artefacts
iso_forest = pickle.load(open('saved_models/isolation_forest.pkl', 'rb'))
autoencoder = load_model('saved_models/autoencoder.keras')
ae_threshold = pickle.load(open('saved_models/ae_threshold.pkl', 'rb'))
scaler_scores = pickle.load(open('saved_models/scaler_scores.pkl', 'rb'))

# Inference on new flows (already preprocessed by Phase 1+2 pipeline)
def predict_anomaly(X_new):
    # Autoencoder
    recon = autoencoder.predict(X_new, verbose=0)
    mse   = np.mean(np.square(X_new - recon), axis=1)
    ae_pred = (mse > ae_threshold).astype(int)

    # Isolation Forest
    if_raw  = iso_forest.predict(X_new)
    if_pred = np.where(if_raw == 1, 0, 1)

    # Ensemble: flag if either model flags (high-recall mode)
    ensemble_pred = np.maximum(ae_pred, if_pred)
    return ae_pred, if_pred, ensemble_pred

ae_p, if_p, ens_p = predict_anomaly(X_test_aug[:10])
print('AE predictions      :', ae_p)
print('IF predictions      :', if_p)
print('Ensemble predictions:', ens_p)
```


---
## Section 8 — Critical Analysis & Recommendations


In [ ]:
# ============================================================
# SECTION 8 : CRITICAL ANALYSIS & FINAL SUMMARY
# ============================================================
print('=' * 70)
print('CRITICAL ANALYSIS OF RESULTS')
print('=' * 70)

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
METHODOLOGICAL FIXES APPLIED IN THIS VERSION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ FIX 1 — No threshold leakage
   Autoencoder threshold is calibrated on X_val_aug (held-out val),
   never on X_test_aug. The test set is touched once, at the end.

✅ FIX 2 — Realistic anomaly injection
   Attack-type-specific perturbations (SYN Flood, Port Scan,
   DDoS Volume, Data Exfiltration) replace uniform Gaussian noise.
   Only the feature subsets relevant to each attack type are perturbed.
   Metrics now reflect realistic detection capability.

✅ FIX 3 — Correct IF contamination strategy
   contamination='auto' is set explicitly and intentionally (not as
   a fallback). The training set is 100% benign by design; 'auto'
   is the documented correct setting for this regime.

✅ FIX 4 — Cross-validation stability
   5-fold StratifiedKFold with per-fold anomaly injection + per-fold
   AE threshold calibration provides variance estimates for all metrics.

✅ FIX 5 — Model persistence
   All artefacts saved: IF (pkl), AE (keras), encoder (keras),
   threshold (pkl), score scaler (pkl), CV results (csv).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REMAINING LIMITATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️  Synthetic anomalies (even structured ones) remain a simplification.
    Evaluation on datasets with real labelled attacks (CIC-IDS-2018,
    UNSW-NB15, TON_IoT) is required for production performance claims.

⚠️  One-class training assumption: if attack flows exist in the raw
    dataset but are unlabelled, training includes them as 'normal',
    which reduces sensitivity to those specific attack patterns.

⚠️  AE threshold depends on the anomaly fraction in the val set.
    In production, recalibrate periodically as traffic baseline drifts.

⚠️  Feature importance (Section 6.7) is correlation-based, not causal.
    SHAP (shap.DeepExplainer / shap.TreeExplainer) provides per-flow
    attribution and should be used for operational explainability.""")

print('\n' + '='*70)
print('FINAL METRICS SUMMARY (held-out test set)')
print('='*70)
print(df_metrics.round(4).to_string())
print()

fn_ae = metrics_ae['FN']; fn_if = metrics_if['FN']
fp_ae = metrics_ae['FP']; fp_if = metrics_if['FP']
n_att = int(y_test_aug.sum())
n_ben = int((y_test_aug == 0).sum())
print(f'  Attacks missed (FN):')
print(f'    Autoencoder      : {fn_ae:,}  ({fn_ae/n_att*100:.1f}% of real attacks)')
print(f'    Isolation Forest : {fn_if:,}  ({fn_if/n_att*100:.1f}% of real attacks)')
print(f'  False alarms (FP):')
print(f'    Autoencoder      : {fp_ae:,}  ({fp_ae/n_ben*100:.1f}% of benign flows)')
print(f'    Isolation Forest : {fp_if:,}  ({fp_if/n_ben*100:.1f}% of benign flows)')

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DEPLOYMENT RECOMMENDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Hybrid cascade architecture (production):
  Tier 1 — Isolation Forest  : fast, low-latency screening
  Tier 2 — Autoencoder       : deep analysis of flagged flows
  Priority metric             : RECALL (missed attack >> false alarm
                                in a 6G smart city critical infra)
"""
)
print('✓ Analysis complete — all artefacts saved to:', MODELS_DIR)
